In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install -q \
datasets \
transformers \
sentence-transformers \
accelerate \
evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


In [3]:
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModel,
    pipeline
)

from sentence_transformers import (
    SentenceTransformer,
    util
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import wandb

In [4]:
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

train = dataset["train"]

print(train)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})


In [5]:
train = train.map(
    lambda x: {
        "combined_text":
            x["prompt"] + " " + x["A"]
    }
)

print(
    len(train[51]["combined_text"])
)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

614


In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
print(tokenizer.vocab_size)

30522


In [8]:
print(
    tokenizer.sep_token_id
)

102


python lists

In [11]:
prompts = list(train["prompt"])
options_A = list(train["A"])
options_B = list(train["B"])
options_C = list(train["C"])
options_D = list(train["D"])
options_E = list(train["E"])
answers = list(train["answer"])

tokenise

In [12]:
encoding = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print("Input IDs Shape:")
print(encoding["input_ids"].shape)

Input IDs Shape:
torch.Size([2000, 128])


bert

In [13]:
model = AutoModel.from_pretrained(
    "bert-base-uncased"
)

model.eval()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

tokenise first prompt

In [15]:
inputs = tokenizer(
    prompts[0],
    return_tensors="pt"
)

In [16]:
with torch.no_grad():
    outputs = model(**inputs)

In [17]:
print(outputs.last_hidden_state.shape)

torch.Size([1, 31, 768])


In [18]:
cls_embedding = outputs.last_hidden_state[0, 0]

print(cls_embedding.shape)

print(
    round(
        cls_embedding[:5].sum().item(),
        4
    )
)

torch.Size([768])
-1.2001


In [ ]:
attention

In [19]:
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [20]:
sentence = "Light-ion fusion is a technique."

inputs = tokenizer(
    sentence,
    return_tensors="pt"
)

tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

print(tokens)

['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']


In [21]:
fusion_index = tokens.index("fusion")

In [22]:
with torch.no_grad():
    outputs = model(**inputs)

In [23]:
last_layer = outputs.attentions[-1]

In [24]:
head0 = last_layer[0, 0]

In [25]:
attention = head0[0, fusion_index]

print(round(attention.item(), 4))

0.1025


minilm embeddings

In [26]:
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [27]:
embeddings = embedder.encode(
    [
        prompts[0],
        options_B[0]
    ],
    convert_to_tensor=True
)

In [28]:
similarity = util.cos_sim(
    embeddings[0],
    embeddings[1]
)

print(
    round(
        similarity.item(),
        4
    )
)

0.7658


utility functions

In [29]:
def map3_single(actual, predictions):
    """
    Computes MAP@3 for one example.
    """

    predictions = predictions[:3]

    if actual in predictions:
        return 1 / (predictions.index(actual) + 1)

    return 0

In [30]:
def compute_map3(actuals, predictions):

    scores = [
        map3_single(a, p)
        for a, p in zip(actuals, predictions)
    ]

    return np.mean(scores)

In [31]:
class TFIDFRanker:

    def __init__(self):

        self.vectorizer = TfidfVectorizer(
            stop_words="english"
        )

    def fit(self, train):

        corpus = []

        for row in train:

            corpus.append(
                row["prompt"]
            )

            corpus.extend([
                row["A"],
                row["B"],
                row["C"],
                row["D"],
                row["E"]
            ])

        self.vectorizer.fit(corpus)

    def rank(self, prompt, options):

        prompt_vec = self.vectorizer.transform(
            [prompt]
        )

        scores = {}

        for label, option in options.items():

            option_vec = self.vectorizer.transform(
                [option]
            )

            score = cosine_similarity(
                prompt_vec,
                option_vec
            )[0,0]

            scores[label] = score

        ranked = sorted(
            scores,
            key=scores.get,
            reverse=True
        )

        return ranked

minilm ranker

In [32]:
class MiniLMRanker:

    def __init__(self,
                 model_name="sentence-transformers/all-MiniLM-L6-v2"):

        self.model = SentenceTransformer(model_name)

    def rank(self, prompt, options):

        texts = [prompt] + list(options.values())

        embeddings = self.model.encode(
            texts,
            convert_to_tensor=True,
            show_progress_bar=False
        )

        prompt_embedding = embeddings[0]
        option_embeddings = embeddings[1:]

        similarities = util.cos_sim(
            prompt_embedding,
            option_embeddings
        )[0]

        labels = list(options.keys())

        scores = {
            labels[i]: similarities[i].item()
            for i in range(len(labels))
        }

        ranked = sorted(
            scores,
            key=scores.get,
            reverse=True
        )

        return ranked, scores

In [33]:
ranker = MiniLMRanker()

options = {
    "A": options_A[0],
    "B": options_B[0],
    "C": options_C[0],
    "D": options_D[0],
    "E": options_E[0]
}

ranking, scores = ranker.rank(
    prompts[0],
    options
)

print(scores)
print(ranking)
print("Correct:", answers[0])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'A': 0.7308082580566406, 'B': 0.7658098340034485, 'C': 0.7940163612365723, 'D': 0.7700757384300232, 'E': 0.7323130369186401}
['C', 'D', 'B', 'E', 'A']
Correct: B


In [34]:

ranker = MiniLMRanker()

predictions = []

for i in range(len(prompts)):

    options = {
        "A": options_A[i],
        "B": options_B[i],
        "C": options_C[i],
        "D": options_D[i],
        "E": options_E[i]
    }

    ranked, _ = ranker.rank(
        prompts[i],
        options
    )

    predictions.append(ranked[:3])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [35]:
minilm_map3 = compute_map3(
    answers,
    predictions
)

print("MiniLM MAP@3:", round(minilm_map3,5))

MiniLM MAP@3: 0.42308


In [36]:
tfidf = TFIDFRanker()

tfidf.fit(train)

In [37]:
tfidf_predictions = []

for i in range(len(prompts)):

    options = {
        "A": options_A[i],
        "B": options_B[i],
        "C": options_C[i],
        "D": options_D[i],
        "E": options_E[i]
    }

    ranked = tfidf.rank(
        prompts[i],
        options
    )

    tfidf_predictions.append(
        ranked[:3]
    )

In [38]:
tfidf_map3 = compute_map3(
    answers,
    tfidf_predictions
)

print(tfidf_map3)

0.3119166666666666


In [39]:
improved = 0

for actual, tfidf_pred, minilm_pred in zip(
    answers,
    tfidf_predictions,
    predictions
):

    tfidf_correct = actual in tfidf_pred
    minilm_correct = actual in minilm_pred

    if (not tfidf_correct) and minilm_correct:
        improved += 1

print(improved)

462


In [43]:
comparison = pd.DataFrame({
    "Model":[
        "Majority Baseline",
        "TF-IDF",
        "MiniLM"
    ],
    "MAP@3":[
        0.42125,
        tfidf_map3,
        minilm_map3
    ]
})

comparison

,Model,MAP@3
0,Majority Baseline,0.421250
1,TF-IDF,0.311917
2,MiniLM,0.423083


In [44]:
comparison_table = wandb.Table(dataframe=comparison)

wandb.log({
    "Model Comparison": comparison_table
})

zeroshot

In [45]:
zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [47]:
result = zero_shot(
    sequences=prompts[1],
    candidate_labels=[
        options_A[1],
        options_B[1],
        options_C[1]
    ]
)

print(result)

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

In [48]:
print("Top Label :", result["labels"][0])
print("Top Score :", round(result["scores"][0],4))

Top Label : Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.
Top Score : 0.4575


In [49]:
softmax_sum = sum(result["scores"])

print("Sum:", softmax_sum)

Sum: 0.9999999701976776


multilabel

In [50]:
result_multi = zero_shot(
    prompts[1],
    candidate_labels=[
        options_A[1],
        options_B[1],
        options_C[1]
    ],
    multi_label=True
)

print(result_multi)

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion 

In [51]:
sigmoid_sum = sum(result_multi["scores"])

difference = abs(
    softmax_sum -
    sigmoid_sum
)

print(round(difference,4))

0.9995


flan-t5

In [53]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer_t5 = AutoTokenizer.from_pretrained(
    "google/flan-t5-small"
)

model_t5 = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-small"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [54]:
query = (
    f"Question: {prompts[0]}. "
    f"Is the correct answer A: {options_A[0]} "
    f"or B: {options_B[0]}? "
    f"Answer with just the letter A or B."
)

print(query)

Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B.


In [55]:
inputs = tokenizer_t5(
    query,
    return_tensors="pt"
)

outputs = model_t5.generate(
    **inputs,
    max_new_tokens=5
)

answer = tokenizer_t5.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

B


In [59]:
import wandb

wandb.login()

True

In [60]:
run = wandb.init(
    entity="mrinal-pandey2905-pes-university",
    project="23f2000333-t22026",
    name="milestone-2",
)

In [61]:
comparison_table = wandb.Table(dataframe=comparison)

wandb.log({
    "Model Comparison": comparison_table
})

In [64]:
wandb.log({

    "bert_vocab_size": tokenizer.vocab_size,

    "sep_token_id": tokenizer.sep_token_id,

    "tfidf_map3": tfidf_map3,

    "minilm_map3": minilm_map3,

    "improved_questions": improved,

    "zero_shot_score": result["scores"][0],

    "softmax_vs_sigmoid_difference": difference

})

In [65]:
wandb.finish()

bert_vocab_size,▁
improved_questions,▁
minilm_map3,▁
sep_token_id,▁
softmax_vs_sigmoid_difference,▁
tfidf_map3,▁
zero_shot_score,▁
bert_vocab_size,30522
improved_questions,462
minilm_map3,0.42308
sep_token_id,102
